# EDGE shift detection

This tutorial shows how to use **EDGE** as an alternative shift-detection method in TOAD. EDGE is a community contribution based on Bathiany et al. (2020) and Terpstra et al. (2025), implemented in TOAD by [Sjoerd Terpstra](https://github.com/TerpstraS). The TOAD implementation has not been peer-reviewed as part of the core TOAD methodology.

EDGE is a 1D edge-detection algorithm for abrupt shifts (Canny 1986; Bathiany et al. 2020; Terpstra et al. 2025):

1. Smooth the time series with a Gaussian filter.
2. Compute the gradient with the Sobel operator.
3. Thin candidate edges with non-maximum suppression.
4. Threshold the gradient to keep significant edges.
5. Score each edge by **abruptness** = segment-mean jump / pooled std.

**Scores:** sigma-normalised abruptness (not ASDETECT's ~`[-1, 1]`). Cluster with **`shift_threshold=4`** for the usual 4σ cut (not `0.5`).

**Needs variability:** abruptness divides by segment spread — flat or near-constant series give little or no signal. NaN time series return zeros (cells with any NaN are skipped by `compute_shifts`).


In [ ]:
import matplotlib.pyplot as plt

from toad import TOAD

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 300

### Test data

We use Northern Hemisphere March sea-ice concentration (`siconc`) from an E3SM 1pctCO2 CMIP6 simulation (`CMIP.E3SM-Project.E3SM-1-0.1pctCO2`). The subset is stored under `tutorials/test_data/`.


In [ ]:
td = TOAD("test_data/CMIP.E3SM-Project.E3SM-1-0.1pctCO2.SImon.gr_NH_march.nc")

In [ ]:
from toad.shifts import EDGE

edge_method = EDGE(
    lmin=None,  # min segment length for abruptness; default 10% of series length
    lmax=None,  # max segment length; default 20% of series length (set both or neither)
    lcutoff=None,  # timesteps skipped each side of edge; default 2% of series length
    alpha=0.4,  # down-weight pooled std when segment variances differ
    smoothing_scale="auto",  # Gaussian smooth before gradient: int timesteps, "auto", or None
    gradient_threshold="relative",  # edge candidacy: "relative" or absolute float
    gradient_threshold_multiplier=0.5,  # relative gradient cut = multiplier × max(|gradient|)
)

In [ ]:
td.compute_shifts(method=edge_method)

In [ ]:
td.compute_clusters(shift_threshold=4)

In [ ]:
fig, ax = td.plot.overview(
    vertical=True,
    ncols=2,
    figsize=(12, 8),
    cluster_ids=range(10),
    height_ratios=[1, 2],
)